# ShallowSeek — Knowledge Distillation for Speculative Decoding

This notebook demonstrates the current project contribution of siyuan zhang - 388143 for the Part II milestone. It is intended to run inside the RCP Jupyter environment launched by `notebooks/submit.sh`.

It covers:

- loading the Hugging Face base draft model and UltraChat dataset;
- the training commands for CE, FKL, RKL, and JSD on UltraChat 10k and 50k;
- the SD evaluation protocol for pretrained and trained drafts;
- loading saved `eval_summary.json` files and producing a compact results table.

By default this notebook is safe to `Run All`: long training/evaluation cells print the commands instead of running them. Set `RUN_TRAINING=True` or `RUN_EVAL=True` in the configuration cell to execute the full pipeline.

## 1. Environment

Start the RCP job from your laptop with:

```bash
MSYS_NO_PATHCONV=1 MSYS2_ARG_CONV_EXCL="*" bash notebooks/submit.sh milestone
runai workspace port-forward <job-name> --port 8888:8888 -p course-cs-552-sizhang
```

Then open JupyterLab at `http://localhost:8888` with token `cs552`.

In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

def find_repo_root(start: Path | None = None) -> Path:
    cur = (start or Path.cwd()).resolve()
    for p in [cur, *cur.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'src' / 'kdsd').exists():
            return p
    raise RuntimeError('Could not find repository root. Open this notebook from inside the repo clone.')

REPO = find_repo_root()
os.chdir(REPO)
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))

os.environ.setdefault('HF_HOME', '/scratch/hf_cache')
os.environ.setdefault('HF_HUB_CACHE', '/scratch/hf_cache/hub')
os.environ.setdefault('HF_DATASETS_CACHE', '/scratch/hf_cache/datasets')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

print('repo:', REPO)
print('HF_HOME:', os.environ['HF_HOME'])


In [ ]:
# Execution switches. Leave these False for a quick grading/demo run.
RUN_TRAINING = False
RUN_EVAL = False
LOAD_BASE_DRAFT_MODEL = True

# Shared experiment protocol.
PROJECT = 'cs552-kdsd'
DATA_ROOT = Path('/scratch/cs552-data')
LOSS_ROOT = Path('/scratch/cs552-loss-ablation')
EVAL_JSONL = DATA_ROOT / 'processed' / 'ultrachat_50k' / 'eval.jsonl'

TRAIN_STEPS = 8000
SAVE_STEPS = 2000
DATA_MAX_SEQ_LEN = 512
TRAIN_BS = 2
GRAD_ACCUM = 4
LR = '2e-5'
SCHEDULER = 'cosine'
WARMUP = '0.03'

EVAL_LIMIT = 50
GAMMA = 4
MAX_NEW_TOKENS = 256
N_WARMUP = 1
N_REPEATS = 3
MODE = 'sampling'
TEMPERATURE = 1.0
TOP_P = 0.9

# Existing training run directories used in our preliminary experiments.
# The historical `m128` suffix is part of the folder name only; the evaluation
# commands below use MAX_NEW_TOKENS=256.
RUN50 = LOSS_ROOT / 'ultra50k_bugfix_s8000_seq512_effbs8_eval50_m128_20260521_140437'
RUN10 = LOSS_ROOT / 'ultra10k_bugfix_s8000_seq512_effbs8_eval50_m128_20260522_094615'
CE50 = LOSS_ROOT / 'ce_ultra50k_bugfix_s8000_seq512_effbs8'
CE10 = LOSS_ROOT / 'ce_ultra10k_bugfix_s8000_seq512_effbs8'

print('RUN_TRAINING =', RUN_TRAINING)
print('RUN_EVAL =', RUN_EVAL)
print('eval prompts =', EVAL_JSONL)


## 2. Hugging Face Dataset and Base Model

The project uses Hugging Face models and datasets directly. This cell loads a small slice of UltraChat and the 0.5B Qwen draft model used as the speculative-decoding draft initialization.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

dataset = load_dataset('HuggingFaceH4/ultrachat_200k', split='train_sft[:2]')
print(dataset)
print('first row keys:', list(dataset[0].keys()))

draft_id = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(draft_id)
print('tokenizer vocab size:', len(tokenizer))

if LOAD_BASE_DRAFT_MODEL:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = AutoModelForCausalLM.from_pretrained(
        draft_id,
        dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        attn_implementation='sdpa',
    ).to(device)
    print('loaded draft model on', device)
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 3. Training Commands

The non-CE runs are driven by `scripts/run_loss_size_ablation.sh` for 50k and `scripts/run_loss_size_ablation_10k.sh` for 10k. CE is trained separately with the same training hyperparameters as the KD models. The only intended difference is `loss=ce` versus `loss=fkl/rkl/jsd`.

In [ ]:
def run_cmd(cmd: str, *, execute: bool = False) -> None:
    print('\n$ ' + cmd)
    if execute:
        subprocess.run(cmd, shell=True, check=True)

common_train = (
    f'data.max_seq_len={DATA_MAX_SEQ_LEN} '
    f'train.max_steps={TRAIN_STEPS} '
    f'train.save_steps={SAVE_STEPS} train.eval_steps={SAVE_STEPS} '
    'train.save_total_limit=4 train.load_best_model_at_end=true '
    'train.metric_for_best_model=eval_loss train.greater_is_better=false '
    'train.save_best_model=true '
    f'train.learning_rate={LR} train.lr_scheduler_type={SCHEDULER} train.warmup_ratio={WARMUP} '
    f'train.per_device_train_batch_size={TRAIN_BS} train.gradient_accumulation_steps={GRAD_ACCUM} '
    'train.per_device_eval_batch_size=4 train.report_to_wandb=false'
)

train_commands = [
    'WANDB=false bash scripts/run_loss_size_ablation.sh',
    'WANDB=false bash scripts/run_loss_size_ablation_10k.sh',
    'mkdir -p /scratch/cs552-ce-train-logs',
    'uv run python scripts/train_size.py run_name=ce_ultra50k_bugfix_s8000_seq512_effbs8 '
    'output_dir=/scratch/cs552-loss-ablation/ce_ultra50k_bugfix_s8000_seq512_effbs8/checkpoints/ce_ultra50k_bugfix_s8000_seq512_effbs8 '
    f'loss=ce data=ultrachat_50k {common_train} '
    'hydra.run.dir=/scratch/cs552-loss-ablation/ce_ultra50k_bugfix_s8000_seq512_effbs8/hydra '
    '2>&1 | tee /scratch/cs552-ce-train-logs/train_ce_ultra50k_bugfix_s8000_seq512_effbs8.log',
    'uv run python scripts/train_size.py run_name=ce_ultra10k_bugfix_s8000_seq512_effbs8 '
    'output_dir=/scratch/cs552-loss-ablation/ce_ultra10k_bugfix_s8000_seq512_effbs8/checkpoints/ce_ultra10k_bugfix_s8000_seq512_effbs8 '
    f'loss=ce data=ultrachat_10k {common_train} '
    'hydra.run.dir=/scratch/cs552-loss-ablation/ce_ultra10k_bugfix_s8000_seq512_effbs8/hydra '
    '2>&1 | tee /scratch/cs552-ce-train-logs/train_ce_ultra10k_bugfix_s8000_seq512_effbs8.log',
]

for cmd in train_commands:
    run_cmd(cmd, execute=RUN_TRAINING)


## 4. SD Evaluation Commands

All models are evaluated on the same clean prompt file, `ultrachat_50k/eval.jsonl`, so 10k and 50k models are directly comparable. The protocol is aligned with the current FKL resolved config: sampling, `gamma=4`, `max_new_tokens=256`, `n_repeats=3`.

In [ ]:
def eval_cmd(run_name: str, draft: str, results_dir: str, log_path: str) -> str:
    return (
        'uv run python scripts/evaluate_sd.py '
        f'run_name={run_name} '
        f'draft={draft} '
        f'prompts.jsonl={EVAL_JSONL} prompts.hf_dataset=null prompts.limit={EVAL_LIMIT} '
        f'runtime.mode={MODE} runtime.temperature={TEMPERATURE} runtime.top_p={TOP_P} '
        f'runtime.gamma={GAMMA} runtime.max_new_tokens={MAX_NEW_TOKENS} '
        f'eval.n_warmup={N_WARMUP} eval.n_repeats={N_REPEATS} '
        f'results_dir={results_dir} '
        f'2>&1 | tee {log_path}'
    )

eval_specs = []
for data_tag, run_dir in [('ultra50k', RUN50), ('ultra10k', RUN10)]:
    for loss in ['fkl', 'rkl', 'jsd']:
        ckpt = run_dir / 'checkpoints' / f'{loss}_{data_tag}_bugfix_s8000_seq512_effbs8_a1_effbs8' / 'model'
        name = f'eval_{loss}_{data_tag}_s8000_seq512_effbs8_a1_g4_eval50_m256_r3_on50keval'
        eval_specs.append((name, str(ckpt), str(run_dir / 'results' / name)))

for data_tag, run_dir in [('ultra50k', CE50), ('ultra10k', CE10)]:
    ckpt = run_dir / 'checkpoints' / f'ce_{data_tag}_bugfix_s8000_seq512_effbs8' / 'model'
    name = f'eval_ce_{data_tag}_s8000_seq512_effbs8_g4_eval50_m256_r3_on50keval'
    eval_specs.append((name, str(ckpt), str(run_dir / 'results' / name)))

eval_specs.append((
    'eval_pretrained_05b_g4_eval50_m256_r3_on50keval',
    'Qwen/Qwen2.5-0.5B-Instruct',
    '/scratch/cs552-results/eval_pretrained_05b_g4_eval50_m256_r3_on50keval',
))

Path('/scratch/cs552-milestone-eval-logs').mkdir(parents=True, exist_ok=True)
for name, draft, out in eval_specs:
    cmd = eval_cmd(name, draft, out, f'/scratch/cs552-milestone-eval-logs/{name}.log')
    run_cmd(cmd, execute=RUN_EVAL)


## 5. Results Table

This cell discovers completed `eval_summary.json` files for the evaluated models and displays the main SD metrics. Missing rows simply mean that the corresponding training/evaluation has not been run yet in the current `/scratch` environment.

In [ ]:
def load_summary(path: Path) -> dict | None:
    if not path.exists():
        return None
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)

rows = []
for name, _draft, out in eval_specs:
    summary_path = Path(out) / 'eval_summary.json'
    s = load_summary(summary_path)
    if s is None:
        rows.append({'run': name, 'status': 'missing', 'summary_path': str(summary_path)})
        continue
    rows.append({
        'run': name,
        'status': 'done',
        'acceptance_rate': s.get('acceptance_rate'),
        'avg_accepted_tokens': s.get('avg_accepted_tokens'),
        'speedup': s.get('speedup'),
        'tokens_per_second': s.get('tokens_per_second'),
        'summary_path': str(summary_path),
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception:
    for row in rows:
        print(row)


## 6. Cached 10k/50k Evaluation Results

Full training and evaluation are too expensive to rerun during a notebook demo. Training the KD drafts takes many GPU hours, and even loading existing checkpoints and running the unified SD evaluation protocol can take more than 4 hours for this set of models. Therefore, this notebook keeps the executable commands above for reproducibility, but directly records the completed evaluation summaries from `10k_50k_results/` for analysis.

All rows below use the same evaluation protocol: `ultrachat_50k/eval.jsonl`, 50 prompts, `gamma=4`, `max_new_tokens=256`, sampling with `temperature=1.0`, `top_p=0.9`, `n_warmup=1`, and `n_repeats=3`.

| model | train data | objective | acceptance rate | avg accepted tokens | speedup | tokens/s | SD time (s) | vanilla time (s) |
|---|---:|---|---:|---:|---:|---:|---:|---:|
| pretrained Qwen2.5-0.5B | none | pretrained | 0.450 | 1.801 | 0.508 | 18.080 | 628.9 | 319.2 |
| CE 10k | UltraChat 10k | CE | 0.423 | 1.694 | 0.486 | 17.748 | 643.2 | 312.5 |
| FKL 10k | UltraChat 10k | FKL | 0.447 | 1.788 | 0.509 | 17.865 | 632.7 | 321.9 |
| RKL 10k | UltraChat 10k | RKL | 0.463 | 1.851 | 0.508 | 18.506 | 622.2 | 315.9 |
| JSD 10k | UltraChat 10k | JSD | 0.469 | 1.876 | 0.524 | 18.849 | 603.3 | 316.1 |
| CE 50k | UltraChat 50k | CE | 0.429 | 1.714 | 0.496 | 17.826 | 634.0 | 314.4 |
| FKL 50k | UltraChat 50k | FKL | 0.451 | 1.804 | 0.509 | 17.687 | 642.2 | 327.2 |
| RKL 50k | UltraChat 50k | RKL | 0.472 | 1.888 | 0.521 | 18.773 | 610.6 | 318.2 |
| JSD 50k | UltraChat 50k | JSD | 0.471 | 1.886 | 0.524 | 18.045 | 630.7 | 330.4 |

Main takeaways from these completed runs: CE-only training underperforms the pretrained draft on acceptance rate, while KD objectives improve draft-target agreement. RKL and JSD are the strongest objectives in both the 10k and 50k settings, with the best acceptance rate from RKL 50k and the best measured HF-loop speedup from JSD 10k/JSD 50k. The wall-clock speedups remain below 1x because this HF instrumented loop exposes detailed acceptance and timing metrics rather than using an optimized production inference engine.

In [ ]:
# Self-contained 10k vs 50k analysis from the completed results above.
# This cell does not read JSON files and can run without executing previous cells.
import matplotlib.pyplot as plt
import pandas as pd

rows = [
    {
        'model': 'pretrained Qwen2.5-0.5B',
        'train_data': 'pretrained',
        'objective': 'pretrained',
        'acceptance_rate': 0.4503466557911909,
        'avg_accepted_tokens': 1.8013866231647635,
        'speedup': 0.5075373747452575,
        'tokens_per_second': 18.079710466387365,
        'sd_time_s': 628.9370629657058,
        'vanilla_time_s': 319.20906581760704,
    },
    {
        'model': 'CE 10k',
        'train_data': '10k',
        'objective': 'CE',
        'acceptance_rate': 0.4234769687964339,
        'avg_accepted_tokens': 1.6939078751857355,
        'speedup': 0.4857984885301736,
        'tokens_per_second': 17.74846967864597,
        'sd_time_s': 643.2103841456897,
        'vanilla_time_s': 312.4706324248884,
    },
    {
        'model': 'FKL 10k',
        'train_data': '10k',
        'objective': 'FKL',
        'acceptance_rate': 0.4469641251940835,
        'avg_accepted_tokens': 1.787856500776334,
        'speedup': 0.5087557905017256,
        'tokens_per_second': 17.864969865772164,
        'sd_time_s': 632.6906837752709,
        'vanilla_time_s': 321.8850489671653,
    },
    {
        'model': 'RKL 10k',
        'train_data': '10k',
        'objective': 'RKL',
        'acceptance_rate': 0.4628702716008862,
        'avg_accepted_tokens': 1.8514810864035447,
        'speedup': 0.5076301159654848,
        'tokens_per_second': 18.505908177155526,
        'sd_time_s': 622.2337152961021,
        'vanilla_time_s': 315.8645730533948,
    },
    {
        'model': 'JSD 10k',
        'train_data': '10k',
        'objective': 'JSD',
        'acceptance_rate': 0.4691166610794502,
        'avg_accepted_tokens': 1.8764666443178009,
        'speedup': 0.5239097536536697,
        'tokens_per_second': 18.84895456464947,
        'sd_time_s': 603.3225853983314,
        'vanilla_time_s': 316.08658708973485,
    },
    {
        'model': 'CE 50k',
        'train_data': '50k',
        'objective': 'CE',
        'acceptance_rate': 0.42853448961731244,
        'avg_accepted_tokens': 1.7141379584692498,
        'speedup': 0.495899984514268,
        'tokens_per_second': 17.825759308531637,
        'sd_time_s': 634.0262877099836,
        'vanilla_time_s': 314.4136262570197,
    },
    {
        'model': 'FKL 50k',
        'train_data': '50k',
        'objective': 'FKL',
        'acceptance_rate': 0.4510233319688907,
        'avg_accepted_tokens': 1.8040933278755629,
        'speedup': 0.5094669917174816,
        'tokens_per_second': 17.68676113062409,
        'sd_time_s': 642.1752358227967,
        'vanilla_time_s': 327.16708555010456,
    },
    {
        'model': 'RKL 50k',
        'train_data': '50k',
        'objective': 'RKL',
        'acceptance_rate': 0.47206447302488724,
        'avg_accepted_tokens': 1.888257892099549,
        'speedup': 0.5211662271273737,
        'tokens_per_second': 18.772881527663206,
        'sd_time_s': 610.5615689903498,
        'vanilla_time_s': 318.2040693396703,
    },
    {
        'model': 'JSD 50k',
        'train_data': '50k',
        'objective': 'JSD',
        'acceptance_rate': 0.47146819060425244,
        'avg_accepted_tokens': 1.8858727624170097,
        'speedup': 0.5237972144475006,
        'tokens_per_second': 18.04471667565907,
        'sd_time_s': 630.7109279998888,
        'vanilla_time_s': 330.36462720793986,
    },
]

objective_order = ['CE', 'FKL', 'RKL', 'JSD']
df = pd.DataFrame(rows)
df['objective'] = pd.Categorical(df['objective'], ['pretrained', *objective_order], ordered=True)
df = df.sort_values(['objective', 'train_data']).reset_index(drop=True)

pretrained = df[df['train_data'] == 'pretrained'].iloc[0]
paired = df[df['train_data'].isin(['10k', '50k'])].copy()
paired['acceptance_delta_vs_pretrained'] = paired['acceptance_rate'] - pretrained['acceptance_rate']
paired['speedup_delta_vs_pretrained'] = paired['speedup'] - pretrained['speedup']

comparison = paired.pivot(index='objective', columns='train_data', values='acceptance_rate').loc[objective_order]
comparison['50k_minus_10k_acceptance'] = comparison['50k'] - comparison['10k']

print('Unified eval protocol: ultrachat_50k/eval.jsonl, 50 prompts, gamma=4, max_new_tokens=256, n_repeats=3')
print('Pretrained acceptance baseline:', round(pretrained['acceptance_rate'], 4))
display(
    paired[
        [
            'objective',
            'train_data',
            'acceptance_rate',
            'avg_accepted_tokens',
            'speedup',
            'tokens_per_second',
            'acceptance_delta_vs_pretrained',
        ]
    ].sort_values(['objective', 'train_data'])
)

display(comparison)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
metrics = [
    ('acceptance_rate', 'Acceptance Rate'),
    ('avg_accepted_tokens', 'Avg Accepted Tokens'),
    ('tokens_per_second', 'Tokens / Second'),
]

for ax, (metric, title) in zip(axes, metrics):
    pivot = paired.pivot(index='objective', columns='train_data', values=metric).loc[objective_order]
    pivot.plot(kind='bar', ax=ax, width=0.75)
    ax.axhline(pretrained[metric], color='black', linestyle='--', linewidth=1, label='pretrained')
    ax.set_title(title)
    ax.set_xlabel('Objective')
    ax.grid(axis='y', alpha=0.25)
    ax.legend()

plt.tight_layout()
plt.show()

best_acceptance = paired.loc[paired['acceptance_rate'].idxmax()]
best_speedup = paired.loc[paired['speedup'].idxmax()]
print(
    f"Best acceptance: {best_acceptance['objective']} {best_acceptance['train_data']} "
    f"({best_acceptance['acceptance_rate']:.3f})."
)
print(
    f"Best HF-loop speedup: {best_speedup['objective']} {best_speedup['train_data']} "
    f"({best_speedup['speedup']:.3f}x)."
)
print(
    'Main pattern: RKL/JSD improve acceptance over CE and pretrained; 50k is slightly better for RKL, '
    'while JSD is already strong at 10k under this eval protocol.'
)

## 7. 10k vs 50k Comparison

Under the unified evaluation protocol, increasing the training data from UltraChat 10k to UltraChat 50k consistently improves acceptance rate for every objective, but the gain is modest:

- CE: `0.423 -> 0.429`
- FKL: `0.447 -> 0.451`
- RKL: `0.463 -> 0.472`
- JSD: `0.469 -> 0.471`

The largest data-size gain appears for RKL, where 50k improves acceptance rate by about `+0.009`. JSD is already strong with 10k examples and only improves slightly with 50k, suggesting that this objective is relatively data-efficient under the current setup.

Overall, the KD objective matters more than the 10k-to-50k data increase. CE-only training underperforms the pretrained draft, FKL is close to the pretrained baseline, and RKL/JSD are the strongest objectives. The best 50k model by acceptance rate is RKL 50k (`0.472`), while the best 10k model is JSD 10k (`0.469`).

## 8. Preliminary Observations

The cached `10k_50k_results/` summaries above show that KD improves draft-target agreement over CE-only training. Under the unified protocol, RKL/JSD are the strongest KD objectives by acceptance rate and average accepted tokens.

All HF-loop wall-clock speedups may remain below 1x because the custom Python/HF instrumentation prioritizes measurable acceptance metrics over optimized inference throughput.